# Estatística — fórmulas e códigos de resolução

**Guia de consulta e prática** organizado por assunto. Cada seção apresenta palavras-chave para reconhecer o tipo de exercício, a fórmula ou o modelo estatístico aplicável, as condições de uso e um bloco Python com entradas identificadas e resultados interpretáveis.

**Como utilizar:** localize o assunto pelo índice ou pelas palavras-chave dos títulos e explicações. Em cada bloco, altere as variáveis indicadas em **ENTRADAS** e execute a célula. As variáveis adicionais estão separadas por finalidade e podem permanecer com os valores de exemplo quando não forem necessárias.

**Atenção às hipóteses:** verifique se o enunciado trata de população ou amostra, se a amostragem é com ou sem reposição, se os eventos são independentes e quais pressupostos são exigidos pelo teste ou modelo. Bibliotecas podem adotar convenções específicas de parametrização.

## Índice
1. Análise de Dados Quantitativos
2. Probabilidades
3. Variáveis Aleatórias Discretas Unidimensionais
4. Variáveis Aleatórias Contínuas Unidimensionais
5. Testes de Hipótese
6. Modelo Básico de Regressão Linear
7. Dicionário de notações

**Bibliotecas necessárias:** `numpy scipy pandas matplotlib statsmodels sympy`. Instale no ambiente Python com `python -m pip install numpy scipy pandas matplotlib statsmodels sympy`.

## Preparação do ambiente

Execute esta célula uma vez no início do notebook. Ela importa as bibliotecas utilizadas nos cálculos estatísticos, distribuições de probabilidade, testes de hipótese, regressão e gráficos.

In [ ]:
import numpy as np
import pandas as pd
from fractions import Fraction
from scipy import stats, integrate, special
import statsmodels.api as sm
import matplotlib.pyplot as plt

print("Bibliotecas carregadas. Pronto para calcular!")

---
# 1. Análise de Dados Quantitativos

**Identificação:** quando a questão fornece observações numéricas e pede resumo, posição, dispersão, comparação ou associação.

### Análise descritiva de dados: média, mediana, moda, amplitude, variância, desvio-padrão, quartis, percentis, frequências e outliers

**Use quando o exercício fornecer dados quantitativos e pedir medidas de tendência central ou dispersão, quartis, percentis, intervalo interquartil (AIQ), coeficiente de variação, escore padronizado (z-score), tabela de frequências ou boxplot para identificar possíveis outliers.**

Informe o conjunto de dados uma única vez. O bloco calcula as medidas principais, frequências absolutas e relativas, e gera um boxplot. Os campos de padronização e média ponderada são opcionais. A variância e o desvio-padrão são apresentados para população e amostra.

In [ ]:
# ENTRADAS — conjunto principal de observações
dados = np.array([2, 4, 4, 6, 9, 10, 12, 15], dtype=float)

# ENTRADAS — opções de percentil e padronização
percentil_desejado = 90
valor_para_padronizar = None
media_referencia = None
desvio_referencia = None

# ENTRADAS — média ponderada (opcional; pesos devem corresponder aos valores)
valores_ponderados = None
pesos = None

# RESULTADOS — medidas de posição e dispersão
modas, contagens = np.unique(dados, return_counts=True)
modas = modas[contagens == contagens.max()]
q1, mediana, q3 = np.percentile(dados, [25, 50, 75], method="linear")
print("Quantidade de observações:", len(dados))
print("Média:", np.mean(dados))
print("Mediana:", mediana)
print("Moda(s):", modas.tolist())
print("Amplitude:", np.ptp(dados))
print("Variância populacional:", np.var(dados, ddof=0))
print("Variância amostral:", np.var(dados, ddof=1) if len(dados)>1 else "indisponível: amostra com menos de dois valores")
print("Desvio-padrão populacional:", np.std(dados, ddof=0))
print("Desvio-padrão amostral:", np.std(dados, ddof=1) if len(dados)>1 else "indisponível: amostra com menos de dois valores")

# RESULTADOS — quartis, percentis e intervalo interquartil
print("\nQuartis e percentis")
print("Q1:", q1, "| Mediana (Q2):", mediana, "| Q3:", q3)
print("Intervalo interquartil (AIQ):", q3-q1)
print(f"Percentil {percentil_desejado}:", np.percentile(dados, percentil_desejado, method="linear"))

# RESULTADOS — coeficiente de variação, escore z e média ponderada
if np.mean(dados)!=0 and len(dados)>1:
    print("\nCoeficiente de variação amostral (%):", np.std(dados,ddof=1)/np.mean(dados)*100)
else:
    print("\nCoeficiente de variação: indisponível (média zero ou amostra insuficiente).")
if valor_para_padronizar is not None and media_referencia is not None and desvio_referencia not in (None,0):
    print("Escore z do valor informado:",(valor_para_padronizar-media_referencia)/desvio_referencia)
elif valor_para_padronizar is not None:
    print("Para calcular z, informe média_referencia e desvio_referencia diferente de zero.")
if valores_ponderados is not None and pesos is not None:
    print("Média ponderada:",np.average(valores_ponderados,weights=pesos))

# RESULTADOS — tabela de frequências e boxplot
frequencias=pd.Series(dados).value_counts().sort_index()
tabela=pd.DataFrame({"frequência absoluta":frequencias,
 "frequência relativa":frequencias/len(dados),
 "frequência acumulada":frequencias.cumsum(),
 "frequência relativa acumulada":(frequencias/len(dados)).cumsum()})
print("\nTabela de frequências")
display(tabela)
plt.figure(figsize=(7,2.5))
plt.boxplot(dados,vert=False)
plt.title("Boxplot dos dados")
plt.xlabel("Valor")
plt.show()

### Covariância e correlação de Pearson

**Quando usar:** quantificar associação linear entre duas variáveis quantitativas.

**Fórmula(s):**

$$s_{xy}=\frac{\sum(x_i-\bar{x})(y_i-\bar{y})}{n-1},\qquad r=\frac{s_{xy}}{s_xs_y}$$

Correlação não implica causalidade; Pearson mede associação **linear**.

In [ ]:
x = np.array([1, 2, 3, 4, 5], dtype=float)
y = np.array([2, 3, 5, 4, 7], dtype=float)
print("Covariância amostral:", np.cov(x, y, ddof=1)[0, 1])
print("Correlação de Pearson:", stats.pearsonr(x, y).statistic)

---
# 2. Probabilidades

**Identificação:** eventos, contagens de casos, independência, condicionamento e atualização de probabilidades.

### Complemento, união e interseção

**Quando usar:** calcular a probabilidade de “não A” ou “A ou B”.

**Fórmula(s):**

$$P(A^c)=1-P(A)$$

$$P(A\cup B)=P(A)+P(B)-P(A\cap B)$$

Se A e B são mutuamente exclusivos, a interseção vale zero.

In [ ]:
p_a, p_b, p_intersecao = 0.4, 0.5, 0.2
print("P(não A):", 1 - p_a)
print("P(A ou B):", p_a + p_b - p_intersecao)

### Probabilidade condicional e independência

**Quando usar:** o enunciado contém “dado que”, “sabendo que” ou pergunta se eventos são independentes.

**Fórmula(s):**

$$P(A\mid B)=\frac{P(A\cap B)}{P(B)}\quad (P(B)>0)$$

$$A\perp B\iff P(A\cap B)=P(A)P(B)$$

Não confundir **independentes** com **mutuamente exclusivos**.

In [ ]:
p_a, p_b, p_ab = 0.4, 0.5, 0.2
print("P(A|B):", p_ab / p_b)
print("São independentes?", np.isclose(p_ab, p_a * p_b))

### Probabilidade total e teorema de Bayes

**Quando usar:** há grupos/categorias mutuamente exclusivos e exaustivos; atualizar a probabilidade após uma evidência.

**Fórmula(s):**

$$P(B)=\sum_iP(B\mid A_i)P(A_i)$$

$$P(A_j\mid B)=\frac{P(B\mid A_j)P(A_j)}{\sum_iP(B\mid A_i)P(A_i)}$$

Os eventos Aᵢ devem formar uma partição e o denominador deve ser positivo.

In [ ]:
prioris = np.array([0.7, 0.3])
verossimilhancas = np.array([0.1, 0.8])  # P(B|A_i)
p_b = np.dot(prioris, verossimilhancas)
posteriores = prioris * verossimilhancas / p_b
print("P(B):", p_b)
print("P(A_i|B):", posteriores)

### Arranjos, permutações e combinações

**Quando usar:** contar resultados possíveis antes de calcular probabilidades.

**Fórmula(s):**

$$n!=n(n-1)\cdots1$$

$$A(n,k)=\frac{n!}{(n-k)!},\qquad \binom nk=\frac{n!}{k!(n-k)!}$$

**Arranjo:** ordem importa. **Combinação:** ordem não importa. **Permutação de n distintos:** n!.

In [ ]:
n, k = 8, 3
print("Permutações de n:", special.factorial(n, exact=True))
print("Arranjos sem repetição:", special.perm(n, k, exact=True))
print("Combinações sem repetição:", special.comb(n, k, exact=True))

---
# 3. Variáveis Aleatórias Discretas Unidimensionais

**Identificação:** valores isolados ou contagens; probabilidades pontuais e acumuladas.

**Atalho de interpretação (distribuições discretas):** `pmf(k)` = exatamente k; `cdf(k)` = até k (inclusive); `cdf(k-1)` = menos que k; `sf(k-1)` = pelo menos k; `sf(k)` = mais que k.

### Função de probabilidade e função acumulada

**Quando usar:** a variável assume valores discretos, com probabilidades associadas.

**Fórmula(s):**

$$p_X(x)=P(X=x),\qquad F_X(x)=P(X\leq x)=\sum_{t\leq x}p_X(t)$$

$$\sum_xp_X(x)=1$$

Verifique que todas as probabilidades são não negativas e somam 1.

In [ ]:
valores = np.array([0, 1, 2, 3])
probs = np.array([0.1, 0.2, 0.4, 0.3])
assert np.all(probs >= 0) and np.isclose(probs.sum(), 1)
print("P(X=2):", probs[valores == 2].sum())
print("P(X<=2):", probs[valores <= 2].sum())

### Esperança, variância e desvio-padrão discretos

**Quando usar:** a questão fornece valores possíveis e probabilidades, sem nomear uma distribuição padrão.

**Fórmula(s):**

$$E(X)=\sum_xxp_X(x),\qquad E(X^2)=\sum_xx^2p_X(x)$$

$$\operatorname{Var}(X)=E(X^2)-[E(X)]^2,\qquad \sigma_X=\sqrt{\operatorname{Var}(X)}$$

Probabilidades podem ser decimais (`0.25`) ou frações Python (`1/4`).

In [ ]:
valores = np.array([0, 1, 2, 3], dtype=float)
probs = np.array([1/10, 2/10, 4/10, 3/10], dtype=float)
assert np.isclose(probs.sum(), 1)
esperanca = np.dot(valores, probs)
variancia = np.dot(valores**2, probs) - esperanca**2
print("Esperança:", esperanca)
print("Variância:", variancia)
print("Desvio-padrão:", np.sqrt(max(variancia, 0)))

### Transformações lineares

**Quando usar:** calcular esperança e variância de Y=aX+b.

**Fórmula(s):**

$$E(aX+b)=aE(X)+b$$

$$\operatorname{Var}(aX+b)=a^2\operatorname{Var}(X)$$

A constante b desloca a média, mas não altera a variância.

In [ ]:
media_x, var_x = 4, 9
a, b = 2, 5
print("E(Y):", a * media_x + b)
print("Var(Y):", a**2 * var_x)

### Bernoulli

**Quando usar:** uma única tentativa, com sucesso (1) ou fracasso (0).

**Fórmula(s):**

$$P(X=x)=p^x(1-p)^{1-x},\quad x\in\{0,1\}$$

$$E(X)=p,\qquad \operatorname{Var}(X)=p(1-p)$$

Parâmetro: probabilidade p de sucesso.

In [ ]:
probabilidade = 0.2

p = probabilidade
d = stats.bernoulli(p)
print("P(X=1):", d.pmf(1), "E(X):", d.mean(), "Var(X):", d.var())

### Binomial

**Quando usar:** n tentativas independentes, duas possibilidades em cada uma e p constante.

**Fórmula(s):**

$$P(X=k)=\binom nkp^k(1-p)^{n-k}$$

$$E(X)=np,\qquad \operatorname{Var}(X)=np(1-p)$$

**Sem reposição** em população finita geralmente não é binomial; veja hipergeométrica.

In [ ]:
instancias = 5
probabilidade = 10/20
instancias_sucesso = 2

n, p, k = instancias, probabilidade, instancias_sucesso
d = stats.binom(n=n, p=p)
print("Exatamente k:", d.pmf(k))
print("No máximo k:", d.cdf(k))
print("Menos que k:", d.cdf(k-1))
print("Pelo menos k:", d.sf(k-1))
print("Mais que k:", d.sf(k))
print("Média:", d.mean())
print("Variância:", d.var())

### Poisson

**Quando usar:** contagem de ocorrências em intervalo, sob modelo de taxa média constante e ocorrências independentes.

**Fórmula(s):**

$$P(X=k)=e^{-\lambda}\frac{\lambda^k}{k!}$$

$$E(X)=\operatorname{Var}(X)=\lambda$$

Se a taxa é por unidade e o intervalo tem duração t, use λ = taxa × t.

In [ ]:
# ENTRADAS — taxa média por unidade de tempo, duração e contagem
taxa_media = 3
duracao_intervalo = 6
k = 8

# RESULTADOS — distribuição de Poisson
lambda_intervalo = taxa_media*duracao_intervalo
distribuicao = stats.poisson(mu=lambda_intervalo)
print("Parâmetro λ para o intervalo:",lambda_intervalo)
print(f"Probabilidade de exatamente {k} ocorrências:",distribuicao.pmf(k))
print(f"Probabilidade de pelo menos {k} ocorrências:",distribuicao.sf(k-1))
print(f"Probabilidade de no máximo {k} ocorrências:",distribuicao.cdf(k))
print("Número esperado de ocorrências:",distribuicao.mean())
print("Variância do número de ocorrências:",distribuicao.var())

### Geométrica

**Quando usar:** contar o número de tentativas até o primeiro sucesso, com ensaios independentes e p constante.

**Fórmula(s):**

$$P(X=k)=(1-p)^{k-1}p,\quad k=1,2,\ldots$$

$$E(X)=\frac1p,\qquad \operatorname{Var}(X)=\frac{1-p}{p^2}$$

**Convenção:** SciPy conta tentativas a partir de 1. Alguns livros contam fracassos a partir de 0.

In [ ]:
p = 0.8
k = 3
d = stats.geom(p)
print(f"Primeiro sucesso na tentativa {k}: {d.pmf(k)}")
print(f"Até {k} tentativas: {d.cdf(k)}")
print(f"Média: {d.mean()}\nVariância: {d.var()}")

print("Sucesso exatamente em k para vários k:")

for k in range(1, k+1):
    print(f"k={k}: {d.pmf(k)}")

### Hipergeométrica

**Quando usar:** amostragem **sem reposição** de população finita, com sucessos e fracassos.

**Fórmula(s):**

$$P(X=k)=\frac{\binom Kk\binom{N-K}{n-k}}{\binom Nn}$$

$$E(X)=n\frac KN$$

N = população; K = sucessos na população; n = tamanho da amostra. SciPy usa `M=N, n=K, N=n_amostra`.

In [ ]:
# ENTRADAS — população, sucessos na população e tamanho da amostra
tamanho_populacao = 50
quantidade_sucessos_populacao = 20
tamanho_amostra = 10
k = 4

# RESULTADOS — distribuição hipergeométrica (amostragem sem reposição)
d = stats.hypergeom(M=tamanho_populacao,n=quantidade_sucessos_populacao,N=tamanho_amostra)
print(f"Probabilidade de exatamente {k} sucessos:",d.pmf(k))
print(f"Probabilidade de no máximo {k} sucessos:",d.cdf(k))
print(f"Probabilidade de menos que {k} sucessos:",d.cdf(k-1))
print(f"Probabilidade de pelo menos {k} sucessos:",d.sf(k-1))
print("Número esperado de sucessos:",d.mean())
print("Variância do número de sucessos:",d.var())

---
# 4. Variáveis Aleatórias Contínuas Unidimensionais

**Identificação:** densidades, integrais, probabilidades de intervalos, percentis e modelos contínuos. Para variável contínua, P(X = a) = 0.

### Densidade, acumulada e intervalos

**Quando usar:** uma função densidade f(x) é fornecida e pede-se probabilidade em intervalo.

**Fórmula(s):**

$$f(x)\geq0,\qquad \int_{-\infty}^{\infty}f(x)\,dx=1$$

$$F(x)=\int_{-\infty}^xf(t)\,dt,\qquad P(a\leq X\leq b)=F(b)-F(a)$$

Limites inclusivos/exclusivos não alteram probabilidades em variáveis contínuas.

In [ ]:
# Exemplo: f(x)=2x em [0,1], zero fora
def f(x):
    return 2*x if 0 <= x <= 1 else 0.0

a, b = 0.2, 0.8
prob, erro_numerico = integrate.quad(f, a, b)
print("P(a<=X<=b):", prob)
print("Integral total:", integrate.quad(f, 0, 1)[0])

### Esperança e variância contínuas

**Quando usar:** a densidade é conhecida e não há distribuição padrão diretamente aplicável.

**Fórmula(s):**

$$E(X)=\int_{-\infty}^{\infty}xf(x)\,dx$$

$$\operatorname{Var}(X)=\int_{-\infty}^{\infty}x^2f(x)\,dx-[E(X)]^2$$

Integração numérica retorna aproximações; confirme o suporte e se f integra 1.

In [ ]:
def f(x):
    return 2*x if 0 <= x <= 1 else 0.0
media = integrate.quad(lambda x: x*f(x), 0, 1)[0]
segundo_momento = integrate.quad(lambda x: x*x*f(x), 0, 1)[0]
print("Esperança:", media)
print("Variância:", segundo_momento - media**2)

### Uniforme contínua

**Quando usar:** todos os pontos de um intervalo [a,b] têm a mesma densidade.

**Fórmula(s):**

$$f(x)=\frac1{b-a},\quad a\leq x\leq b$$

$$E(X)=\frac{a+b}{2},\qquad \operatorname{Var}(X)=\frac{(b-a)^2}{12}$$

SciPy: `loc=a`, `scale=b-a` (não é o extremo superior).

In [ ]:
a, b = 2, 8
d = stats.uniform(loc=a, scale=b-a)
print("P(3<=X<=5):", d.cdf(5) - d.cdf(3))
print("Média, variância:", d.mean(), d.var())

### Normal e padronização Z

**Quando usar:** valores contínuos aproximadamente normais; cálculo de caudas e percentis.

**Fórmula(s):**

$$X\sim N(\mu,\sigma^2),\qquad Z=\frac{X-\mu}{\sigma}\sim N(0,1)$$

$$f(x)=\frac{1}{\sigma\sqrt{2\pi}}e^{-\frac{(x-\mu)^2}{2\sigma^2}}$$

SciPy: `scale` recebe **desvio-padrão σ**, não variância σ². `ppf(q)` é o quantil com probabilidade acumulada q.

In [ ]:
mu, sigma, x = 100, 15, 120
d = stats.norm(loc=mu, scale=sigma)
print("P(X<=x):", d.cdf(x))
print("P(X>x):", d.sf(x))
print("P(90<=X<=120):", d.cdf(120) - d.cdf(90))
print("Percentil 95:", d.ppf(0.95))
print("Z de x:", (x-mu)/sigma)

### Exponencial

**Quando usar:** tempo de espera entre eventos de processo de Poisson com taxa λ.

**Fórmula(s):**

$$f(x)=\lambda e^{-\lambda x},\quad x\geq0$$

$$F(x)=1-e^{-\lambda x},\quad E(X)=\frac1\lambda,\quad \operatorname{Var}(X)=\frac1{\lambda^2}$$

SciPy: `scale=1/lambda` (não passar λ diretamente como scale).

In [ ]:
taxa, t = 0.5, 3
D = stats.expon(scale=1/taxa)
print("P(X<=t):", D.cdf(t))
print("P(X>t):", D.sf(t))
print("Média, variância:", D.mean(), D.var())

### Quantis e probabilidades inversas

**Quando usar:** a questão fornece uma probabilidade e pede o valor de corte.

**Fórmula(s):**

$$F(x_q)=q\qquad\Longrightarrow\qquad x_q=F^{-1}(q)$$

`ppf(q)` é o inverso da acumulada; `isf(q)` encontra o corte cuja cauda superior vale q.

In [ ]:
d = stats.norm(loc=50, scale=10)
print("Valor abaixo do qual estão 90%:", d.ppf(0.90))
print("Valor com 5% acima:", d.isf(0.05))

---
# 5. Testes de Hipótese

**Roteiro obrigatório:** (1) escrever H₀ e H₁; (2) identificar parâmetro e pressupostos; (3) escolher teste e cauda; (4) calcular estatística e valor-p; (5) comparar com α; (6) concluir **no contexto**. **Não rejeitar H₀** não significa demonstrar que H₀ é verdadeira.

**Regra:** rejeitar H₀ quando `valor-p <= alfa`. Use teste unilateral somente quando a hipótese alternativa for direcional **antes** de olhar os dados.

### Teste Z para média (σ populacional conhecido)

**Quando usar:** testar média populacional com σ conhecido e população normal ou aproximação normal justificada.

**Fórmula(s):**

$$H_0:\mu=\mu_0,\qquad Z=\frac{\bar X-\mu_0}{\sigma/\sqrt n}$$

Escolha `alternativa` = `"two-sided"`, `"greater"` ou `"less"`.

In [ ]:
media_amostral, mu0, sigma_pop, n = 53, 50, 10, 40
alternativa, alfa = "two-sided", 0.05
z = (media_amostral - mu0) / (sigma_pop / np.sqrt(n))
p = (2*stats.norm.sf(abs(z)) if alternativa == "two-sided" else
     stats.norm.sf(z) if alternativa == "greater" else stats.norm.cdf(z))
print("Estatística Z:",z)
print("Valor-p:",p)
print("Nível de significância α:",alfa)
print("Decisão:","Rejeitar H₀" if p<=alfa else "Não rejeitar H₀")

### Teste t para uma média (σ desconhecido)

**Quando usar:** testar média com desvio-padrão populacional desconhecido; independência e normalidade/aproximação adequadas.

**Fórmula(s):**

$$T=\frac{\bar X-\mu_0}{s/\sqrt n},\qquad gl=n-1$$

`ttest_1samp` aceita observações brutas e calcula estatística, valor-p e graus de liberdade.

In [ ]:
dados = np.array([49, 52, 54, 51, 55, 53, 50, 56], dtype=float)
mu0, alternativa, alfa = 50, "greater", 0.05
res = stats.ttest_1samp(dados, popmean=mu0, alternative=alternativa)
print("Estatística t:",res.statistic)
print("Valor-p:",res.pvalue)
print("Nível de significância α:",alfa)
print("Decisão:","Rejeitar H₀" if res.pvalue<=alfa else "Não rejeitar H₀")

### Intervalo de confiança para média

**Quando usar:** estimar média com incerteza (não é teste por si só).

**Fórmula(s):**

$$IC_{1-\alpha}(\mu)=\bar x\pm t_{1-\alpha/2,\,n-1}\frac{s}{\sqrt n}$$

Forma t para σ desconhecido; pressupostos análogos ao teste t de uma amostra.

In [ ]:
dados = np.array([49, 52, 54, 51, 55, 53, 50, 56], dtype=float)
confianca = 0.95
n = len(dados)
media, se = np.mean(dados), stats.sem(dados)
intervalo = stats.t.interval(confianca, df=n-1, loc=media, scale=se)
print("IC para média:", intervalo)

### Teste t para duas amostras independentes

**Quando usar:** comparar médias de dois grupos independentes.

**Fórmula(s):**

$$H_0:\mu_1=\mu_2$$

$$t_{Welch}=\frac{\bar x_1-\bar x_2}{\sqrt{s_1^2/n_1+s_2^2/n_2}}$$

**Welch** (`equal_var=False`) não exige variâncias populacionais iguais; exige independência e condições apropriadas.

In [ ]:
grupo1 = np.array([10, 11, 12, 9, 13], dtype=float)
grupo2 = np.array([8, 9, 10, 11, 9], dtype=float)
res = stats.ttest_ind(grupo1, grupo2, equal_var=False, alternative="two-sided")
print("t de Welch:", res.statistic, "valor-p:", res.pvalue)

### Teste t pareado

**Quando usar:** mesmos indivíduos antes/depois ou observações em pares.

**Fórmula(s):**

$$d_i=x_i-y_i,\qquad T=\frac{\bar d}{s_d/\sqrt n}$$

As diferenças entre pares devem satisfazer as condições do teste t de uma amostra.

In [ ]:
antes = np.array([80, 75, 90, 85, 70], dtype=float)
depois = np.array([78, 73, 87, 82, 69], dtype=float)
res = stats.ttest_rel(antes, depois, alternative="two-sided")
print("t pareado:", res.statistic, "valor-p:", res.pvalue)

### Teste para uma proporção (aproximação Z)

**Quando usar:** testar proporção populacional em ensaios Bernoulli independentes com aproximação normal válida.

**Fórmula(s):**

$$H_0:p=p_0,\qquad Z=\frac{\hat p-p_0}{\sqrt{p_0(1-p_0)/n}}$$

Para amostras pequenas, prefira o **teste binomial exato**; abaixo ambos são mostrados.

In [ ]:
sucessos, n, p0 = 56, 100, 0.5
p_hat = sucessos / n
z = (p_hat-p0) / np.sqrt(p0*(1-p0)/n)
p_z = 2*stats.norm.sf(abs(z))
p_exato = stats.binomtest(sucessos, n, p0, alternative="two-sided").pvalue
print("Proporção observada na amostra:",p_hat)
print("Estatística Z:",z)
print("Valor-p pela aproximação normal:",p_z)
print("Valor-p pelo teste binomial exato:",p_exato)

### Qui-quadrado de aderência

**Quando usar:** comparar frequências observadas com frequências esperadas por um modelo categórico.

**Fórmula(s):**

$$\chi^2=\sum_i\frac{(O_i-E_i)^2}{E_i}$$

Frequências esperadas não devem ser muito pequenas; se parâmetros forem estimados dos dados, os graus de liberdade podem mudar.

In [ ]:
observadas = np.array([18, 22, 30, 30])
esperadas = np.array([25, 25, 25, 25])
res = stats.chisquare(f_obs=observadas, f_exp=esperadas)
print("Qui-quadrado:", res.statistic, "valor-p:", res.pvalue)

### Qui-quadrado de independência

**Quando usar:** verificar associação entre duas variáveis categóricas em tabela de contingência.

**Fórmula(s):**

$$E_{ij}=\frac{(\text{total da linha i})(\text{total da coluna j})}{n}$$

$$\chi^2=\sum_{i,j}\frac{(O_{ij}-E_{ij})^2}{E_{ij}}$$

Exige observações independentes; para tabela 2×2 com contagens pequenas, considere Fisher exato.

In [ ]:
tabela = np.array([[20, 10], [12, 18]])
qui2, p, gl, esperadas = stats.chi2_contingency(tabela, correction=False)
print("Qui-quadrado:", qui2, "gl:", gl, "valor-p:", p)
print("Frequências esperadas:\n", esperadas)
print("Fisher exato (2x2):", stats.fisher_exact(tabela).pvalue)

### Erros tipo I, tipo II e poder

**Quando usar:** interpretar decisões e riscos de testes.

**Fórmula(s):**

$$\alpha=P(\text{rejeitar }H_0\mid H_0\text{ verdadeira})$$

$$\beta=P(\text{não rejeitar }H_0\mid H_1\text{ verdadeira}),\qquad \text{poder}=1-\beta$$

Poder depende de tamanho amostral, efeito real, variabilidade e regra do teste. Não há função universal sem especificar esses elementos.

In [ ]:
alfa = 0.05
beta_exemplo = 0.20
print("Nível de significância:", alfa)
print("Poder ilustrativo:", 1-beta_exemplo)

---
# 6. Modelo Básico de Regressão Linear

**Identificação:** relação entre variável explicativa x e resposta y; ajuste por mínimos quadrados; interpretação de coeficientes, resíduos, R² e inferência.

**Pressupostos usuais para inferência OLS:** linearidade na média condicional, exogeneidade, observações adequadamente independentes, variância constante para erros-padrão clássicos e normalidade dos erros para testes exatos em pequenas amostras. Verifique os pressupostos exigidos no curso.

### Regressão linear simples: ajuste, coeficientes, predição, resíduos, R² e inferência

**Use quando o exercício fornecer pares de observações (x, y) e pedir reta de regressão, mínimos quadrados, intercepto, inclinação, previsão, valores ajustados, resíduos, soma dos quadrados dos erros (SQE), coeficiente de determinação R², erro-padrão residual, teste t dos coeficientes ou intervalos de confiança e previsão.**

Informe os vetores de dados uma única vez. O bloco ajusta o modelo OLS com intercepto e apresenta os principais resultados. Altere x_novo para prever a resposta em outro valor de x. R² descreve o ajuste dentro da amostra e não demonstra causalidade. A inferência depende dos pressupostos do modelo indicados na introdução da seção.

In [ ]:
# ENTRADAS — observações da variável explicativa x e da resposta y
x = np.array([1,2,3,4,5,6],dtype=float)
y = np.array([2,3,5,4,6,8],dtype=float)

# ENTRADAS — valor para previsão e nível de confiança
x_novo = 7
confianca = 0.95

# RESULTADOS — ajuste por mínimos quadrados ordinários (OLS)
modelo = sm.OLS(y,sm.add_constant(x)).fit()
intercepto,inclinacao = modelo.params
print("MODELO AJUSTADO")
print(f"Equação estimada: ŷ = {intercepto:.6g} + ({inclinacao:.6g})x")
print("Intercepto:",intercepto)
print("Inclinação (variação estimada em y por unidade de x):",inclinacao)

# RESULTADOS — qualidade do ajuste, resíduos e previsão
print("\nQUALIDADE DO AJUSTE")
print("Coeficiente de determinação R²:",modelo.rsquared)
print("R² ajustado:",modelo.rsquared_adj)
print("Soma dos quadrados dos erros (SQE):",np.sum(modelo.resid**2))
print("Erro-padrão residual:",np.sqrt(modelo.mse_resid))
print("\nVALORES OBSERVADOS, AJUSTADOS E RESÍDUOS")
display(pd.DataFrame({"x observado":x,"y observado":y,"y estimado":modelo.fittedvalues,"resíduo":modelo.resid}))
print(f"Previsão para x = {x_novo}:",modelo.predict(np.array([[1,x_novo]]))[0])

# RESULTADOS — testes t e intervalos de confiança dos coeficientes
print("\nINFERÊNCIA DOS COEFICIENTES")
display(pd.DataFrame({"coeficiente":modelo.params,"erro-padrão":modelo.bse,
 "estatística t":modelo.tvalues,"valor-p":modelo.pvalues},
 index=["Intercepto","Inclinação"]))
print(f"Intervalos de confiança de {confianca:.0%}:")
display(pd.DataFrame(modelo.conf_int(alpha=1-confianca),
 index=["Intercepto","Inclinação"],columns=["limite inferior","limite superior"]))

# RESULTADOS — intervalo da média e intervalo de previsão individual
predicao=modelo.get_prediction(exog=np.array([[1.0,x_novo]]))
print(f"Intervalos para x = {x_novo}:")
display(predicao.summary_frame(alpha=1-confianca))

---
# 7. Dicionário de notações

> **Importante:** uma mesma letra pode representar coisas diferentes conforme o assunto. A interpretação deve considerar a fórmula em que aparece.

## Letras gregas

| Símbolo | Nome | Significado frequente |
|---|---|---|
| $\alpha$ | alfa | nível de significância; também parâmetro em alguns modelos |
| $\beta$ | beta | probabilidade de erro tipo II; coeficiente de regressão (depende do contexto) |
| $\beta_0,\beta_1$ | beta zero, beta um | intercepto e inclinação populacionais |
| $\varepsilon$ | épsilon | termo de erro na regressão |
| $\lambda$ | lambda | taxa ou média da distribuição de Poisson; taxa da exponencial |
| $\mu$ | mi | média populacional / esperança |
| $\pi$ | pi | constante matemática ≈ 3,14159; por vezes proporção populacional |
| $\rho$ | rô | correlação populacional |
| $\sigma$ | sigma | desvio-padrão populacional |
| $\sigma^2$ | sigma ao quadrado | variância populacional |
| $\Sigma$ | sigma maiúsculo | somatório |
| $\chi^2$ | qui-quadrado | estatística ou distribuição qui-quadrado |

## Letras latinas e símbolos estatísticos

| Símbolo | Significado frequente |
|---|---|
| $X,Y$ | variáveis aleatórias |
| $x_i,y_i$ | valores observados |
| $n$ | tamanho da amostra ou número de tentativas |
| $N$ | tamanho da população (ou parâmetro de modelo específico) |
| $k$ | contagem de sucessos ou índice |
| $p$ | probabilidade de sucesso ou proporção populacional |
| $\hat p$ | proporção amostral |
| $\bar x$ | média amostral |
| $s,s^2$ | desvio-padrão e variância amostrais |
| $E(X)$ | esperança / valor esperado |
| $\operatorname{Var}(X)$ | variância de X |
| $f(x)$ | densidade (contínua) ou função definida no contexto |
| $p_X(x)$ | função de probabilidade (discreta) |
| $F(x)$ | função de distribuição acumulada |
| $P(A)$ | probabilidade do evento A |
| $P(A\mid B)$ | probabilidade de A dado B |
| $H_0,H_1$ | hipóteses nula e alternativa |
| $EP$ | erro-padrão |
| $gl$ | graus de liberdade |
| $r$ | correlação amostral |
| $R^2$ | coeficiente de determinação |
| $\hat y$ | valor previsto/ajustado de y |
| $e_i$ | resíduo amostral |
| $\hat\theta$ | estimativa do parâmetro $\theta$ |

## Operadores e relações

| Símbolo | Significado |
|---|---|
| $\sum$ | somatório |
| $\prod$ | produtório |
| $\int$ | integral |
| $\sqrt{a}$ | raiz quadrada |
| $n!$ | fatorial |
| $\binom nk$ | combinações de n elementos tomados k a k |
| $\cap$ | interseção: A **e** B |
| $\cup$ | união: A **ou** B (inclusive) |
| $A^c$ | complemento de A |
| $\mid$ | “dado que”, em probabilidade condicional |
| $\perp$ | independência (quando usado entre eventos/variáveis) |
| $\in$ | pertence |
| $\sim$ | segue a distribuição indicada |
| $\hat{\ }$ | acento chapéu: estimativa ou valor ajustado |
| $\le,\ge$ | menor/maior ou igual |
| $\approx$ | aproximadamente igual |
| $\infty$ | infinito |

## Atalhos Python (SciPy)

| Função | Interpretação |
|---|---|
| `d.pmf(k)` | P(X = k), para distribuições discretas |
| `d.pdf(x)` | densidade em x, **não** probabilidade pontual |
| `d.cdf(x)` | P(X ≤ x) |
| `d.sf(x)` | P(X > x); costuma ser mais preciso que `1 - cdf(x)` |
| `d.ppf(q)` | quantil com acumulada q |
| `d.isf(q)` | quantil da cauda superior q |
| `d.mean()` | esperança |
| `d.var()` | variância |
| `d.std()` | desvio-padrão |
| `np.mean(dados)` | média aritmética dos dados (não é esperança ponderada arbitrária) |
| `np.dot(valores, probabilidades)` | esperança discreta personalizada |

## Checklist de resolução

1. **Identifique o problema:** dados brutos, eventos, distribuição, teste ou regressão?
2. **Confira as hipóteses:** independência, reposição, normalidade, amostragem, variância conhecida?
3. **Defina o que pede:** exatamente, até, pelo menos, intervalo, quantil, média, valor-p?
4. **Ajuste os parâmetros** na célula do tópico correspondente.
5. **Interprete** o número obtido (probabilidade, percentual, unidades, decisão estatística).

**Nota:** códigos exibem aproximações numéricas. Em relatórios ou soluções formais, apresente também a fórmula, a substituição dos valores e a interpretação do resultado.